In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/IPL_Performance_Prediction/data/processed/'
matches = pd.read_csv(base_path + 'matches_cleaned.csv')
deliveries = pd.read_csv(base_path + 'deliveries_cleaned.csv')

matches['date'] = pd.to_datetime(matches['date'])

if 'date' in deliveries.columns:
    deliveries.drop(columns=['date'], inplace=True)

deliveries['total_runs'] = deliveries['batsman_runs'] + deliveries['extras']

# Merge to get Date and Venue info inside the Ball-by-Ball data
df = matches.merge(deliveries, on='match_id')

print(f"Merged Dataset Shape: {df.shape}")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Merged Dataset Shape: (278205, 47)
      year                  venue                  event  winner_runs  \
0  2007/08  M Chinnaswamy Stadium  Indian Premier League        140.0   
1  2007/08  M Chinnaswamy Stadium  Indian Premier League        140.0   
2  2007/08  M Chinnaswamy Stadium  Indian Premier League        140.0   
3  2007/08  M Chinnaswamy Stadium  Indian Premier League        140.0   
4  2007/08  M Chinnaswamy Stadium  Indian Premier League        140.0   

       umpire2                  toss_winner       date neutralvenue  \
0  RE Koertzen  Royal Challengers Bangalore 2008-04-18          NaN   
1  RE Koertzen  Royal Challengers Bangalore 2008-04-18          NaN   
2  RE Koertzen  Royal Challengers Bangalore 2008-04-18          NaN   
3  RE Koertzen  Royal Challengers Bangalore 2008-04-18          NaN   
4  RE Koertzen  Royal Challengers Bangalor

In [ ]:
batter_stats = df.groupby(['batsman', 'date', 'match_id'])['batsman_runs'].sum().reset_index()

batter_stats = batter_stats.sort_values(by=['batsman', 'date'])

# --- FEATURE 1: Batting Form ---
# Calculate average runs in the last 10 innings
batter_stats['avg_runs_last_10'] = batter_stats.groupby('batsman')['batsman_runs'].transform(lambda x: x.rolling(window=10, min_periods=1).mean().shift(1))

# --- FEATURE 2: Batting Class ---
# Average runs over entire career up to the previous match
batter_stats['career_avg_runs'] = batter_stats.groupby('batsman')['batsman_runs'].transform(lambda x: x.expanding().mean().shift(1))

batter_stats.fillna(0, inplace=True)

print("Batting features calculated using 'batsman' column!")
print(batter_stats.tail())

Batting features calculated using 'batsman' column!
      batsman       date  match_id  batsman_runs  avg_runs_last_10  \
17633  Z Khan 2016-04-10    980903             4               4.2   
17634  Z Khan 2016-05-15    980993             2               4.5   
17635  Z Khan 2017-04-08   1082595             1               4.7   
17636  Z Khan 2017-05-06   1082635             2               2.7   
17637  Z Khan 2017-05-14   1082646             1               1.8   

       career_avg_runs  
17633         4.863636  
17634         4.826087  
17635         4.708333  
17636         4.560000  
17637         4.461538  


In [ ]:
valid_dismissals = ['caught', 'bowled', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']

df['is_bowler_wicket'] = df['dismissal_kind'].apply(lambda x: 1 if x in valid_dismissals else 0)

# --- 1. Aggregate Wickets per Match ---
bowler_stats = df.groupby(['bowler', 'date', 'match_id'])['is_bowler_wicket'].sum().reset_index()
bowler_stats.rename(columns={'is_bowler_wicket': 'wickets_in_match'}, inplace=True)

bowler_stats = bowler_stats.sort_values(by=['bowler', 'date'])

# --- 2. Calculate Bowling Features ---
bowler_stats['avg_wickets_last_10'] = bowler_stats.groupby('bowler')['wickets_in_match'].transform(lambda x: x.rolling(window=10, min_periods=1).mean().shift(1))

# --- 3. Career Average Wickets
bowler_stats['career_avg_wickets'] = bowler_stats.groupby('bowler')['wickets_in_match'].transform(lambda x: x.expanding().mean().shift(1))

bowler_stats.fillna(0, inplace=True)

print("Bowling features calculated!")
print(bowler_stats.tail())

Bowling features calculated!
               bowler       date  match_id  wickets_in_match  \
13841  Zeeshan Ansari 2025-04-23   1473478                 1   
13842  Zeeshan Ansari 2025-04-25   1473480                 0   
13843  Zeeshan Ansari 2025-05-02   1473488                 1   
13844  Zeeshan Ansari 2025-05-05   1473492                 0   
13845  Zeeshan Ansari 2025-05-19   1473499                 0   

       avg_wickets_last_10  career_avg_wickets  
13841             0.800000            0.800000  
13842             0.833333            0.833333  
13843             0.714286            0.714286  
13844             0.750000            0.750000  
13845             0.666667            0.666667  


In [ ]:
# --- 1. Venue Stats ---
# Calculate average score at each venue (High scoring vs Low scoring grounds)
match_totals = df.groupby(['match_id', 'venue'])['total_runs'].sum().reset_index()
venue_stats = match_totals.groupby('venue')['total_runs'].mean().reset_index()
venue_stats.rename(columns={'total_runs': 'venue_avg_score'}, inplace=True)

# --- 2. Final Merge ---
# Merge Venue info
df = df.merge(venue_stats, on='venue', how='left')

# Merge Batting Stats (Matching on 'batsman')
final_df = df.merge(batter_stats[['batsman', 'match_id', 'avg_runs_last_10', 'career_avg_runs']], on=['batsman', 'match_id'], how='left')

# Merge Bowling Stats (Matching on 'bowler')
final_df = final_df.merge(bowler_stats[['bowler', 'match_id', 'avg_wickets_last_10', 'career_avg_wickets']], on=['bowler', 'match_id'], how='left')

# Save the final model-ready data
output_path = '/content/drive/MyDrive/IPL_Performance_Prediction/data/processed/model_data.csv'
final_df.to_csv(output_path, index=False)

print(f"Feature Engineering Complete. Final Data Saved to: {output_path}")
print("\nSample Data (Checking Features):")
print(final_df[['date', 'batsman', 'avg_runs_last_10', 'career_avg_runs', 'bowler', 'avg_wickets_last_10', 'career_avg_wickets']].sample(5))

Feature Engineering Complete. Final Data Saved to: /content/drive/MyDrive/IPL_Performance_Prediction/data/processed/model_data.csv

Sample Data (Checking Features):
             date          batsman  avg_runs_last_10  career_avg_runs  \
13287  2008-06-01         SK Raina              26.5        29.076923   
212387 2022-04-09  Abhishek Sharma              12.8        11.954545   
167497 2019-03-31         MS Dhoni              27.8        25.459119   
29543  2010-03-19       TM Dilshan              20.4        23.434783   
184754 2020-10-10     Shubman Gill              33.0        23.642857   

           bowler  avg_wickets_last_10  career_avg_wickets  
13287    SK Warne                  1.3            1.357143  
212387   DJ Bravo                  1.6            1.139073  
167497    S Gopal                  1.2            1.166667  
29543   JA Morkel                  1.0            1.148148  
184754  CJ Jordan                  1.1            0.923077  
